# Observatorio de puntualidad de Rodalies - analisis exploratorio

Este cuaderno explora el historico de retrasos capturado por el proyecto.

**De donde salen los datos.** Cada minuto el ingestor consulta el feed
GTFS-Realtime de Renfe y guarda una fila por cada tren y parada sobre la que la
compania publica informacion. De todas las veces que se observa una misma
parada, la capa analitica se queda con la ultima, que es la mejor estimacion del
retraso realmente sufrido.

**Como ejecutarlo.** Si hay una base de datos del proyecto accesible, el cuaderno
lee de ella. Si no, cae automaticamente a una muestra **sintetica** incluida en
el repositorio, para que se pueda ejecutar nada mas clonarlo. La procedencia se
indica siempre en la salida: ningun grafico se presenta como real si no lo es.

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

plt.rcParams.update(
    {
        "figure.figsize": (11, 4.5),
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

UMBRAL_PUNTUAL_S = 180  # mismo valor que analytics.setting en la base de datos

## 1. Carga de datos

Una sola funcion decide el origen: base de datos si la hay, muestra sintetica en
caso contrario. Asi el cuaderno es reproducible tanto para quien tiene el stack
levantado como para quien solo ha clonado el repositorio.

In [ ]:
CONSULTA = """
SELECT service_date, linea, stop_id, estacion, trip_id, stop_sequence,
       scheduled_arrival, arrival_time, delay_s, schedule_relationship, source
  FROM analytics.mv_stop_final
 WHERE source = 'renfe'
"""

MUESTRA = "../data/sample/observaciones_muestra.csv.gz"


def cargar():
    """Devuelve (dataframe, origen). Intenta la base de datos y si no, la muestra."""
    url = os.environ.get("RODALIES_DATABASE_URL")
    if url:
        try:
            import psycopg

            with psycopg.connect(url, connect_timeout=5) as conn:
                cursor = conn.execute(CONSULTA)
                columnas = [c.name for c in cursor.description]
                datos = pd.DataFrame(cursor.fetchall(), columns=columnas)
            if len(datos) > 0:
                return datos, "base de datos (datos reales de Renfe)"
            print("La base de datos responde pero aun no tiene historico real.")
        except Exception as error:
            print(f"No se ha podido leer de la base de datos ({error}).")

    return pd.read_csv(MUESTRA), "muestra SINTETICA del repositorio"


datos, origen = cargar()

# La procedencia acompana a CADA grafico. Un lector que se encuentre una
# figura suelta tiene que poder saber si mira Renfe o una simulacion.
ES_SINTETICO = "SINT" in origen.upper()
ETIQUETA = "DATOS SINTETICOS - no son cifras reales" if ES_SINTETICO else "datos reales de Renfe"

print(f"Origen: {origen}")
print(f"{len(datos):,} observaciones")
datos.head()

In [ ]:
# Tipos y columnas derivadas. La hora se toma del horario PROGRAMADO: un tren muy
# retrasado tiene que seguir contando en la franja en la que deberia haber pasado.
datos["service_date"] = pd.to_datetime(datos["service_date"]).dt.date
datos["scheduled_arrival"] = pd.to_datetime(datos["scheduled_arrival"], utc=True)
datos["arrival_time"] = pd.to_datetime(datos["arrival_time"], utc=True)
datos["delay_s"] = pd.to_numeric(datos["delay_s"], errors="coerce")

local = datos["scheduled_arrival"].dt.tz_convert("Europe/Madrid")
datos["hora"] = local.dt.hour
datos["dia_semana"] = local.dt.dayofweek
datos["laborable"] = datos["dia_semana"] < 5
datos["retraso_min"] = datos["delay_s"] / 60
datos["puntual"] = datos["delay_s"] <= UMBRAL_PUNTUAL_S

print(f"Periodo: {datos['service_date'].min()} a {datos['service_date'].max()}")
print(f"Lineas: {sorted(datos['linea'].dropna().unique())}")
print(f"Estaciones: {datos['stop_id'].nunique()}")
print(f"Trenes distintos: {datos['trip_id'].nunique():,}")

## 2. Como se distribuye el retraso

La primera pregunta no es cuanto se retrasan los trenes, sino **como** se
reparten los retrasos: es lo que decide que estadistico tiene sentido usar
despues.

In [ ]:
retrasos = datos["delay_s"].dropna()

resumen = pd.Series(
    {
        "observaciones": len(retrasos),
        "media (s)": retrasos.mean(),
        "mediana (s)": retrasos.median(),
        "p90 (s)": retrasos.quantile(0.90),
        "p99 (s)": retrasos.quantile(0.99),
        "maximo (s)": retrasos.max(),
        "% puntuales": 100 * (retrasos <= UMBRAL_PUNTUAL_S).mean(),
        "% mas de 15 min": 100 * (retrasos > 900).mean(),
    }
)
resumen.round(1)

In [ ]:
fig, (izq, der) = plt.subplots(1, 2, figsize=(12, 4))

recorte = retrasos.clip(-120, 1200) / 60
izq.hist(recorte, bins=60, color="#4C78A8", edgecolor="white", linewidth=0.4)
izq.axvline(
    UMBRAL_PUNTUAL_S / 60,
    color="#E45756",
    linestyle="--",
    label=f"umbral de puntualidad ({UMBRAL_PUNTUAL_S // 60} min)",
)
izq.set_xlabel("retraso (minutos, recortado a 20)")
izq.set_ylabel("observaciones")
izq.set_title(f"Distribucion del retraso\n{ETIQUETA}", fontsize=11)
izq.legend()

positivos = retrasos[retrasos > 0] / 60
if len(positivos) > 0:
    bordes = np.logspace(np.log10(0.5), np.log10(max(positivos.max(), 2.0)), 40)
    der.hist(positivos, bins=bordes, color="#F58518", edgecolor="white", linewidth=0.4)
    der.set_xscale("log")
der.set_xlabel("retraso (minutos, escala logaritmica)")
der.set_title(f"La cola es lo que duele\n{ETIQUETA}", fontsize=11)

plt.tight_layout()
plt.show()

La distribucion tiene **cola larga a la derecha**: la mayoria de los trenes
llegan cerca de su hora y un punado se va muy arriba. Por eso el proyecto guarda
mediana y percentiles ademas de la media: un solo tren parado una hora desplaza
la media de toda una linea y el indicador deja de describir lo que vive el
viajero medio.

La metrica honesta para comunicar es el **porcentaje de puntualidad**; la util
para detectar problemas es el **P90**.

## 3. Que lineas acumulan retraso

In [ ]:
por_linea = (
    datos.groupby("linea")
    .agg(
        observaciones=("delay_s", "size"),
        trenes=("trip_id", "nunique"),
        retraso_medio_s=("delay_s", "mean"),
        mediana_s=("delay_s", "median"),
        p90_s=("delay_s", lambda s: s.quantile(0.90)),
        pct_puntualidad=("puntual", lambda s: 100 * s.mean()),
    )
    .sort_values("pct_puntualidad")
    .round(1)
)
por_linea

In [ ]:
orden = por_linea.sort_values("pct_puntualidad")
fig, ejes = plt.subplots(figsize=(10, 4))
colores = [
    "#E45756" if v < 85 else "#F58518" if v < 92 else "#54A24B" for v in orden["pct_puntualidad"]
]
ejes.barh(orden.index.astype(str), orden["pct_puntualidad"], color=colores)
ejes.set_xlabel(f"% de paradas con retraso menor o igual a {UMBRAL_PUNTUAL_S // 60} min")
ejes.set_title(f"Puntualidad por linea\n{ETIQUETA}", fontsize=11)
ejes.set_xlim(0, 100)
for y, valor in enumerate(orden["pct_puntualidad"]):
    ejes.text(valor + 1, y, f"{valor:.1f}%", va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 4. Franja horaria: donde y cuando duele

El mapa de linea contra hora es el grafico que responde a la pregunta practica
del viajero: *si tengo que coger la R3, a que hora no debo cogerla*.

In [ ]:
matriz = datos.pivot_table(index="linea", columns="hora", values="delay_s", aggfunc="mean")
matriz = matriz.reindex(columns=range(5, 24)) / 60

fig, ejes = plt.subplots(figsize=(12, 0.55 * len(matriz) + 2))
imagen = ejes.imshow(matriz.values, aspect="auto", cmap="YlOrRd", origin="lower")
ejes.set_xticks(range(len(matriz.columns)))
ejes.set_xticklabels(matriz.columns)
ejes.set_yticks(range(len(matriz.index)))
ejes.set_yticklabels(matriz.index.astype(str))
ejes.set_xlabel("hora programada (Europe/Madrid)")
ejes.set_title(f"Retraso medio en minutos por linea y franja horaria\n{ETIQUETA}", fontsize=11)
ejes.grid(False)
fig.colorbar(imagen, ax=ejes, label="minutos")
plt.tight_layout()
plt.show()

matriz.round(1)

In [ ]:
por_hora = datos.groupby("hora").agg(
    retraso_medio_min=("retraso_min", "mean"),
    p90_min=("retraso_min", lambda s: s.quantile(0.90)),
    observaciones=("delay_s", "size"),
)

fig, ejes = plt.subplots(figsize=(11, 4))
ejes.plot(por_hora.index, por_hora["retraso_medio_min"], marker="o", color="#4C78A8", label="media")
ejes.plot(
    por_hora.index,
    por_hora["p90_min"],
    marker="o",
    linestyle="--",
    color="#E45756",
    label="percentil 90",
)
ejes.set_xlabel("hora del dia")
ejes.set_ylabel("retraso (minutos)")
ejes.set_title(f"El retraso sigue el ciclo de la demanda\n{ETIQUETA}", fontsize=11)
ejes.set_xticks(range(0, 24, 2))
ejes.legend()
plt.tight_layout()
plt.show()

## 5. Propagacion a lo largo del recorrido

Un tren que sale tarde rara vez recupera. Si el retraso medio crece con la
posicion de la parada dentro del recorrido, el problema no esta en una estacion
concreta sino en la explotacion de la linea entera, y eso cambia por completo la
lectura del dato.

In [ ]:
if "stop_sequence" in datos.columns and datos["stop_sequence"].notna().any():
    posicion = datos.dropna(subset=["stop_sequence"]).copy()
    posicion["stop_sequence"] = posicion["stop_sequence"].astype(int)
else:
    # Sin stop_sequence (por ejemplo en la muestra), se reconstruye el orden por
    # la hora programada dentro de cada tren.
    posicion = datos.dropna(subset=["scheduled_arrival"]).copy()
    posicion["stop_sequence"] = (
        posicion.groupby("trip_id")["scheduled_arrival"].rank(method="first").astype(int)
    )

propagacion = (
    posicion[posicion["stop_sequence"] <= 10]
    .groupby("stop_sequence")["retraso_min"]
    .agg(["mean", "median", "count"])
)

fig, ejes = plt.subplots(figsize=(10, 4))
ejes.plot(propagacion.index, propagacion["mean"], marker="o", color="#B279A2", label="media")
ejes.plot(
    propagacion.index,
    propagacion["median"],
    marker="s",
    linestyle="--",
    color="#72B7B2",
    label="mediana",
)
ejes.set_xlabel("posicion de la parada dentro del recorrido")
ejes.set_ylabel("retraso (minutos)")
ejes.set_title(f"El retraso se acumula segun avanza el trayecto\n{ETIQUETA}", fontsize=11)
ejes.legend()
plt.tight_layout()
plt.show()

# Pendiente por minimos cuadrados: cuantos segundos se pierden por parada.
x = propagacion.index.to_numpy(dtype=float)
y = propagacion["mean"].to_numpy(dtype=float)
pendiente, corte = np.polyfit(x, y, 1)
print(f"Se acumulan {pendiente * 60:.0f} segundos de retraso por cada parada recorrida.")
print(f"Retraso estimado en origen: {corte * 60:.0f} segundos.")

## 6. Estaciones y dias de la semana

In [ ]:
por_estacion = (
    datos.groupby(["stop_id", "estacion"])
    .agg(
        observaciones=("delay_s", "size"),
        retraso_medio_min=("retraso_min", "mean"),
        p90_min=("retraso_min", lambda s: s.quantile(0.90)),
        pct_puntualidad=("puntual", lambda s: 100 * s.mean()),
    )
    .query("observaciones >= 20")  # sin minimo, el ranking es ruido
    .sort_values("retraso_medio_min", ascending=False)
    .round(1)
)
por_estacion.head(12)

In [ ]:
DIAS = ["lunes", "martes", "miercoles", "jueves", "viernes", "sabado", "domingo"]

por_dia = datos.groupby("dia_semana").agg(
    retraso_medio_min=("retraso_min", "mean"),
    pct_puntualidad=("puntual", lambda s: 100 * s.mean()),
    observaciones=("delay_s", "size"),
)
por_dia.index = [DIAS[i] for i in por_dia.index]

fig, ejes = plt.subplots(figsize=(10, 4))
colores = ["#4C78A8"] * 5 + ["#9D755D"] * 2
ejes.bar(por_dia.index, por_dia["retraso_medio_min"], color=colores[: len(por_dia)])
ejes.set_ylabel("retraso medio (minutos)")
ejes.set_title(f"Laborables frente a fin de semana\n{ETIQUETA}", fontsize=11)
plt.tight_layout()
plt.show()

por_dia.round(1)

In [ ]:
laborables = datos.loc[datos["laborable"], "delay_s"].dropna()
findes = datos.loc[~datos["laborable"], "delay_s"].dropna()

if len(laborables) > 30 and len(findes) > 30:
    # Diferencia de medias con el error tipico de la diferencia. No es un
    # contraste formal, pero da idea de si la brecha es real o es ruido.
    diferencia = laborables.mean() - findes.mean()
    error = np.sqrt(laborables.var(ddof=1) / len(laborables) + findes.var(ddof=1) / len(findes))
    print(f"Laborables: {laborables.mean():.0f} s de media ({len(laborables):,} obs.)")
    print(f"Fin de semana: {findes.mean():.0f} s de media ({len(findes):,} obs.)")
    print(
        f"Diferencia: {diferencia:.0f} s  (error tipico {error:.0f} s, "
        f"{abs(diferencia / error):.1f} errores tipicos)"
    )
else:
    print("No hay observaciones suficientes en ambos grupos para comparar.")

## 7. Evolucion del historico

El grafico mas importante del proyecto a medio plazo no es ninguno de los
anteriores: es este. Mide si la captura sigue viva. Un hueco aqui es un dia de
datos que no se recupera.

In [ ]:
por_dia_natural = datos.groupby("service_date").agg(
    observaciones=("delay_s", "size"),
    pct_puntualidad=("puntual", lambda s: 100 * s.mean()),
)

fig, arriba = plt.subplots(figsize=(11, 4))
arriba.bar(
    por_dia_natural.index, por_dia_natural["observaciones"], color="#BAB0AC", label="observaciones"
)
arriba.set_ylabel("observaciones por dia")
arriba.tick_params(axis="x", rotation=45)

abajo = arriba.twinx()
abajo.plot(
    por_dia_natural.index,
    por_dia_natural["pct_puntualidad"],
    color="#E45756",
    marker="o",
    label="puntualidad",
)
abajo.set_ylabel("% puntualidad")
abajo.set_ylim(0, 100)
abajo.grid(False)

arriba.set_title(f"Volumen capturado y puntualidad, dia a dia\n{ETIQUETA}", fontsize=11)
plt.tight_layout()
plt.show()

por_dia_natural.round(1)

In [ ]:
# La procedencia manda sobre la lectura de los resultados.
if ES_SINTETICO:
    print(
        "AVISO: este cuaderno se ha ejecutado sobre la muestra SINTETICA del "
        "repositorio. Los patrones que se ven (hora punta, propagacion, cola de "
        "incidencias) son los que programa el generador, no hallazgos sobre "
        "Rodalies. Las conclusiones de abajo describen el METODO, no la red real. "
        "Para conclusiones reales hay que ejecutarlo contra la base de datos con "
        "historico capturado."
    )
else:
    print(
        f"Conclusiones sobre datos reales de Renfe: {len(datos):,} observaciones "
        f"entre {datos['service_date'].min()} y {datos['service_date'].max()}."
    )

## 8. Conclusiones y limitaciones

**Lo que se ve en los datos**

1. El retraso no se reparte de forma simetrica: tiene cola larga. Comunicar la
   media sola es enganoso; el porcentaje de puntualidad y el P90 describen mejor
   la experiencia real.
2. El retraso sigue el ciclo de la demanda, con maximos en las dos horas punta.
3. El retraso se acumula a lo largo del recorrido: las paradas finales de una
   linea larga heredan lo que ha pasado antes.
4. Las lineas largas de cercanias sufren mas que las cortas y urbanas.

**Limitaciones honestas**

* Se mide **lo que Renfe publica**, no lo que ocurre. Si la compania deja de
  informar de un tren, ese tren no aparece; no es lo mismo que no llevara retraso.
* El feed informa sobre todo de **llegadas**, no de salidas, asi que "retraso"
  significa aqui retraso de llegada a la parada.
* Un tren **suprimido** aparece como parada `SKIPPED`, sin retraso asociado. En
  terminos de experiencia del viajero una supresion es peor que cualquier
  retraso, y esta metrica no la captura: hay que mirarla aparte.
* La hora programada se deduce del propio feed (hora prevista menos retraso).
  Si Renfe cambiara el horario a media jornada, la referencia cambia con el.
* Estas conclusiones valen para el periodo capturado. Con pocas semanas de
  historico, un solo dia con una incidencia grave puede inclinar cualquier
  agregado.

**Siguiente paso natural**

Con varios meses acumulados, la pregunta interesante deja de ser descriptiva y
pasa a ser predictiva: *dado un tren que sale de origen con X minutos de retraso,
cuanto llevara al llegar a Barcelona*. Los datos para responderla se estan
acumulando desde el primer dia.